# OOD Evaluation from Saved Artifacts

This notebook runs out-of-domain evaluation by loading artifacts exported from `Modeling_Evaluation.ipynb`.

It does **not** rerun Part 1 feature extraction/training.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

print('Imports loaded.')

In [ ]:
# Mount Drive (Colab) and set paths
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/BeyondFK'
except Exception:
    BASE_DIR = '.'

RESULTS_DIR = os.path.join(BASE_DIR, 'results')
ARTIFACTS_DIR = os.path.join(RESULTS_DIR, 'artifacts')
OOD_DIR = os.path.join(BASE_DIR, 'ood')
OOD_RESULTS_DIR = os.path.join(RESULTS_DIR, 'ood')
os.makedirs(OOD_RESULTS_DIR, exist_ok=True)

print('BASE_DIR      :', BASE_DIR)
print('ARTIFACTS_DIR :', ARTIFACTS_DIR)
print('OOD_DIR       :', OOD_DIR)
print('OOD_RESULTS   :', OOD_RESULTS_DIR)

In [ ]:
# Load exported artifacts from File 1
with open(os.path.join(ARTIFACTS_DIR, 'feature_schema.json'), 'r') as f:
    schema = json.load(f)

le = joblib.load(os.path.join(ARTIFACTS_DIR, 'label_encoder.joblib'))
selector_static = joblib.load(os.path.join(ARTIFACTS_DIR, 'selector_static.joblib'))
model_static = joblib.load(os.path.join(ARTIFACTS_DIR, 'model_static.joblib'))

# Optional (if COMBO artifacts were exported)
selector_combo_path = os.path.join(ARTIFACTS_DIR, 'selector_combo.joblib')
model_combo_path = os.path.join(ARTIFACTS_DIR, 'model_combo.joblib')
has_combo = os.path.exists(selector_combo_path) and os.path.exists(model_combo_path)
if has_combo:
    selector_combo = joblib.load(selector_combo_path)
    model_combo = joblib.load(model_combo_path)

print('Artifacts loaded. has_combo =', has_combo)
print('Static feature count:', len(schema.get('static_columns', [])))
print('Prompt feature count:', len(schema.get('prompt_columns', [])))

In [ ]:
# Input files (provide precomputed OOD feature CSVs)
# Required columns:
# - STATIC run: all columns listed in schema['static_columns'] + education_level
# - COMBO run : static + prompt columns + education_level
OOD_FILES = {
    'OneStopEnglish': os.path.join(OOD_DIR, 'onestopenglish_features.csv'),
    'UniversalCEFR': os.path.join(OOD_DIR, 'universalcefr_features.csv'),
}

LABEL_COL = schema.get('label_col', 'education_level')


def evaluate_dataset(df, dataset_name):
    out = {}

    y_true = le.transform(df[LABEL_COL])

    # STATIC
    X_static = df[schema['static_columns']].copy()
    X_static_sel = selector_static.transform(X_static)
    y_pred_static = model_static.predict(X_static_sel)

    out['static_macro_f1'] = float(f1_score(y_true, y_pred_static, average='macro'))
    out['static_report'] = classification_report(
        y_true, y_pred_static, target_names=list(le.classes_), output_dict=True
    )

    # Optional COMBO
    if has_combo and schema.get('combo_columns'):
        X_combo = df[schema['combo_columns']].copy()
        X_combo_sel = selector_combo.transform(X_combo)
        y_pred_combo = model_combo.predict(X_combo_sel)
        out['combo_macro_f1'] = float(f1_score(y_true, y_pred_combo, average='macro'))
        out['combo_report'] = classification_report(
            y_true, y_pred_combo, target_names=list(le.classes_), output_dict=True
        )

    out['n_samples'] = int(len(df))
    out['dataset'] = dataset_name
    return out

all_results = {}
summary_rows = []

for name, path in OOD_FILES.items():
    if not os.path.exists(path):
        print(f'[skip] Missing file: {path}')
        continue

    df_ood = pd.read_csv(path)
    missing = [c for c in schema['static_columns'] + [LABEL_COL] if c not in df_ood.columns]
    if missing:
        print(f'[skip] {name} missing required columns (first 10 shown): {missing[:10]}')
        continue

    result = evaluate_dataset(df_ood, name)
    all_results[name] = result

    summary_rows.append({
        'dataset': name,
        'n_samples': result['n_samples'],
        'static_macro_f1': result['static_macro_f1'],
        'combo_macro_f1': result.get('combo_macro_f1', np.nan),
    })

    print(f"{name}: STATIC={result['static_macro_f1']:.4f}" + (f" | COMBO={result['combo_macro_f1']:.4f}" if 'combo_macro_f1' in result else ''))